In [44]:
from qiskit import QuantumCircuit
from qiskit.primitives import StatevectorSampler
import qiskit.quantum_info as qi

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

sampler = StatevectorSampler()
result = sampler.run([qc], shots=1024).result()
print(result[0].data.meas.get_counts())

{'00': 527, '11': 497}


In [45]:
qc.draw()

┌───┐      ░ ┌─┐   
   q_0: ┤ H ├──■───░─┤M├───
        └───┘┌─┴─┐ ░ └╥┘┌─┐
   q_1: ─────┤ X ├─░──╫─┤M├
             └───┘ ░  ║ └╥┘
meas: 2/══════════════╩══╩═
                      0  1

In [71]:
# Build Oracle block
n=8

oracle = QuantumCircuit(n+1, name='U_f')

for i in range(n):
    oracle.cx(n,i)

Uf = oracle.to_gate()
oracle.draw()

┌───┐                                   
q_0: ┤ X ├───────────────────────────────────
     └─┬─┘┌───┐                              
q_1: ──┼──┤ X ├──────────────────────────────
       │  └─┬─┘┌───┐                         
q_2: ──┼────┼──┤ X ├─────────────────────────
       │    │  └─┬─┘┌───┐                    
q_3: ──┼────┼────┼──┤ X ├────────────────────
       │    │    │  └─┬─┘┌───┐               
q_4: ──┼────┼────┼────┼──┤ X ├───────────────
       │    │    │    │  └─┬─┘┌───┐          
q_5: ──┼────┼────┼────┼────┼──┤ X ├──────────
       │    │    │    │    │  └─┬─┘┌───┐     
q_6: ──┼────┼────┼────┼────┼────┼──┤ X ├─────
       │    │    │    │    │    │  └─┬─┘┌───┐
q_7: ──┼────┼────┼────┼────┼────┼────┼──┤ X ├
       │    │    │    │    │    │    │  └─┬─┘
q_8: ──■────■────■────■────■────■────■────■──

In [72]:
DJ = QuantumCircuit(n+1, n)

    # Step 1: make the base circuit

DJ.h(range(n)) # Our system of qubits
DJ.x(n)
DJ.h(n)  # Our indicator qubit
    # Step 2: use the oracle
# Apply the oracle Uf
DJ.append(Uf, range(n+1))

    # Step 3: another hadamard on all Qubits
DJ.h(range(n))

In [73]:
DJ.draw()

┌───┐     ┌──────┐┌───┐
q_0: ┤ H ├─────┤0     ├┤ H ├
     ├───┤     │      │├───┤
q_1: ┤ H ├─────┤1     ├┤ H ├
     ├───┤     │      │├───┤
q_2: ┤ H ├─────┤2     ├┤ H ├
     ├───┤     │      │├───┤
q_3: ┤ H ├─────┤3     ├┤ H ├
     ├───┤     │      │├───┤
q_4: ┤ H ├─────┤4 U_f ├┤ H ├
     ├───┤     │      │├───┤
q_5: ┤ H ├─────┤5     ├┤ H ├
     ├───┤     │      │├───┤
q_6: ┤ H ├─────┤6     ├┤ H ├
     ├───┤     │      │├───┤
q_7: ┤ H ├─────┤7     ├┤ H ├
     ├───┤┌───┐│      │└───┘
q_8: ┤ X ├┤ H ├┤8     ├─────
     └───┘└───┘└──────┘     
c: 8/═══════════════════════

In [74]:
# How do we interpret a result?

DJ.measure(range(n), range(n))

In [75]:
from qiskit import transpile
from qiskit_aer import AerSimulator

simulator = AerSimulator()
compiled = transpile(DJ, simulator)

result = simulator.run(compiled, shots=1024).result()
counts = result.get_counts()

print(counts)

{'00000000': 1024}
